In [1]:
from typing import TypedDict

class User(TypedDict):
    id: int
    name: str
    email: str

user1: User = {
    'id': 1,
    'name': 'nayeon_park',
    'email': 'example@gmail.com'
}
print(user1)

{'id': 1, 'name': 'nayeon_park', 'email': 'example@gmail.com'}


In [4]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str

user_data = {
    'id': 1,
    'name': '123',
    'email': 'example@gmail.com'
}
user1 = User(**user_data)
print(user1)

id=1 name='123' email='example@gmail.com'


In [5]:
from typing import TypedDict, Annotated

def add(left, right):
    return left + right

class State(TypedDict):
    messages: Annotated[list[str], add]

In [6]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hi there!", id="2")]

add_messages(msgs1, msgs2)

[HumanMessage(content='Hello', additional_kwargs={}, response_metadata={}, id='1'),
 AIMessage(content='Hi there!', additional_kwargs={}, response_metadata={}, id='2', tool_calls=[], invalid_tool_calls=[])]

In [7]:
msgs1 = [HumanMessage(content="Hello", id="1")]
msgs2 = [AIMessage(content="Hello again!", id="1")]

add_messages(msgs1, msgs2)

[AIMessage(content='Hello again!', additional_kwargs={}, response_metadata={}, id='1', tool_calls=[], invalid_tool_calls=[])]

In [8]:
from langchain_core.messages import AnyMessage
from langgraph.graph.message import add_messages
from typing import TypedDict, Annotated

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [2]:
from typing import TypedDict, Annotated
from operator import add

from langgraph.graph import StateGraph

class State(TypedDict):
    messages: Annotated[list[str], add]

graph = StateGraph(State)

def chatbot(state: State):
    question = state["messages"]
    answer = f"사용자 입력을 그대로 반환하는 챗봇입니다. {question}라는 질문을 받았습니다."
    return {"messages": [answer]}

graph.add_node("chatbot", chatbot)

In [3]:
from langgraph.graph import START, END

graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

In [4]:
graph.add_edge("node_a", "node_b")

In [5]:
def routing_function(state: State):
    if len(state["messages"][-1]) > 1000 :
        return True
    return False
graph.add_conditional_edges(
    "chatbot",
    routing_function,
    {True: "Summary", False: END}
)